# 🏭 Projeto de Análise de Dados: Manutenção Preditiva Industrial
### Redução de Falhas Inesperadas em Máquinas de Produção

**Analista responsável:** *Rafael Estefânio De Melo*

**Data:** (25/08/26)

**Área solicitante:** Diretoria de Operações
**Fonte dos dados:** Kaggle — [AI4I 2020 Predictive Maintenance Dataset](https://www.kaggle.com/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020) (arquivo `ai4i2020.csv`)
**Documento de referência:** `Briefing_Projeto_Manutencao_Preditiva.docx`

---
> 💡 Este notebook já está estruturado com todas as etapas do projeto. Seu trabalho é escrever o código de análise em cada célula marcada com `# Seu código aqui`, interpretar os resultados e preencher as conclusões de cada seção.


## 1. Contexto do Negócio

A empresa opera uma linha de máquinas de usinagem (fresadoras) em ambiente industrial. Nos últimos meses, paradas não programadas por falha de equipamento vêm gerando atrasos na produção e custos extras de manutenção corretiva.

A Diretoria de Operações solicitou uma análise exploratória dos dados históricos de sensores e ocorrências de falha para entender **por que as máquinas falham**, **quais condições antecedem uma falha** e **o que pode ser feito para preveni-las**.

Consulte o documento `Briefing_Projeto_Manutencao_Preditiva.docx` para o problema de negócio completo, objetivos e todas as perguntas a serem respondidas.


## 2. Objetivos do Projeto

**Objetivo geral:** Identificar os principais fatores associados às falhas das máquinas e propor recomendações de manutenção preditiva/preventiva baseadas em dados.

**Objetivos específicos:**
- Descrever o comportamento geral das variáveis operacionais e da taxa de falhas.
- Investigar a relação entre variáveis de processo (temperatura, torque, velocidade, desgaste da ferramenta) e a ocorrência de falhas.
- Comparar o comportamento entre os diferentes tipos de falha (TWF, HDF, PWF, OSF, RNF).
- Traduzir os achados técnicos em recomendações de negócio, priorizando o monitoramento das variáveis mais relevantes.


## 3. Perguntas de Negócio a Responder

1. Qual é a taxa geral de falhas das máquinas no período analisado?
2. Quais tipos de falha (TWF, HDF, PWF, OSF, RNF) são mais frequentes?
3. Existe diferença na taxa de falha entre os diferentes tipos/qualidade de produto (L, M, H)?
4. Como a temperatura do ar e a temperatura do processo se relacionam com a ocorrência de falhas?
5. A diferença entre temperatura do processo e temperatura do ar (delta térmico) influencia falhas por dissipação de calor (HDF)?
6. Qual a relação entre velocidade rotacional e torque em máquinas que falharam vs. que não falharam?
7. O desgaste da ferramenta (tool wear) está associado a maior probabilidade de falha?
8. Existe um limiar de desgaste da ferramenta a partir do qual o risco de falha aumenta significativamente?
9. Quais combinações de variáveis operacionais mais aparecem em falhas por sobrecarga (OSF)?
10. As falhas aleatórias (RNF) apresentam algum padrão identificável ou são realmente imprevisíveis?
11. Se cada falha gera, em média, X horas de máquina parada, qual o impacto estimado no período analisado?
12. Quais variáveis têm maior correlação com a ocorrência de falha geral?
13. É possível segmentar as máquinas em grupos de risco (baixo, médio, alto) com base nas variáveis disponíveis?
14. Quais recomendações de manutenção preventiva podem ser feitas com base nos padrões encontrados?
15. Se a empresa pudesse monitorar de perto apenas 2 ou 3 variáveis para antecipar falhas, quais seriam e por quê?

> As respostas devem compor o relatório final, com gráficos, insights e recomendações para a Diretoria.


## 4. Importação das Bibliotecas

In [82]:
# Manipulação de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Configurações gerais de visualização
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

pd.set_option("display.max_columns", None)


## 5. Carregamento dos Dados

**Fonte dos dados:** Kaggle — *AI4I 2020 Predictive Maintenance Dataset*
Link: https://www.kaggle.com/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020
Arquivo: `ai4i2020.csv` (10.000 registros, 14 colunas, dados sintéticos que simulam um processo real de usinagem industrial).

> Baixe o arquivo `ai4i2020.csv` do Kaggle e coloque-o na mesma pasta deste notebook (ou ajuste o caminho abaixo).


In [83]:
# Carregando o dataset
# Fonte: https://www.kaggle.com/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020
caminho: str = "ai4i2020_original.csv"
df = pd.read_csv(caminho)
df.head()


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## 6. Dicionário de Dados

| Coluna | Descrição |
|---|---|
| `UDI` | Identificador único do registro (1 a 10.000) |
| `Product ID` | Identificador do produto (letra de qualidade + número de série) |
| `Type` | Categoria de qualidade do produto: **L** (Low/baixa, ~50%), **M** (Medium/média, ~30%), **H** (High/alta, ~20%) |
| `Air temperature [K]` | Temperatura do ar (Kelvin), gerada por passeio aleatório em torno de 300 K |
| `Process temperature [K]` | Temperatura do processo (Kelvin), correlacionada com a temperatura do ar + ~10 K |
| `Rotational speed [rpm]` | Velocidade rotacional da ferramenta (rpm) |
| `Torque [Nm]` | Torque aplicado (Newton-metro), distribuído normalmente em torno de 40 Nm |
| `Tool wear [min]` | Tempo de desgaste da ferramenta (minutos) |
| `Machine failure` | Variável-alvo: **1** = houve falha da máquina, **0** = não houve falha |
| `TWF` | Falha por desgaste de ferramenta (*Tool Wear Failure*) |
| `HDF` | Falha por dissipação de calor (*Heat Dissipation Failure*) |
| `PWF` | Falha de potência (*Power Failure*) |
| `OSF` | Falha por sobrecarga (*Overstrain Failure*) |
| `RNF` | Falha aleatória, sem causa determinística (*Random Failure*) |

> `TWF`, `HDF`, `PWF`, `OSF` e `RNF` são flags binárias (0/1) que indicam qual(is) modo(s) de falha ocorreu(ram) quando `Machine failure = 1`.


## 7. Exploração Inicial e Qualidade dos Dados

Antes de responder às perguntas de negócio, avalie a estrutura geral do dataset: tipos de dados, valores ausentes, duplicados e estatísticas descritivas.


In [84]:
# Estrutura geral do dataset
df.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9)

In [85]:
# Estatísticas descritivas das variáveis numéricas
df.describe()


,UDI,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
count,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,5000.50000,300.004930,310.005560,1538.776100,39.986910,107.951000,0.033900,0.004600,0.011500,0.009500,0.009800,0.00190
std,2886.89568,2.000259,1.483734,179.284096,9.968934,63.654147,0.180981,0.067671,0.106625,0.097009,0.098514,0.04355
min,1.00000,295.300000,305.700000,1168.000000,3.800000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,2500.75000,298.300000,308.800000,1423.000000,33.200000,53.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
50%,5000.50000,300.100000,310.100000,1503.000000,40.100000,108.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
75%,7500.25000,301.500000,311.100000,1612.000000,46.800000,162.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
max,10000.00000,304.500000,313.800000,2886.000000,76.600000,253.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000


In [86]:
# Verificação de valores ausentes e duplicados
print(f"Valores ausentes:\n{df.isnull().sum()}\n")
print("_" * 50)
print(f"Valores duplicados: {df.duplicated().sum()}\n")


Valores ausentes:
UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

__________________________________________________
Valores duplicados: 0



**Insight:**
- *Não achamos valores nulos e nem valores duplicados para trata-los!*
- *A base tem um valor total de:* 10000 registros.

## 8. Análises por Pergunta de Negócio

### Pergunta 1
**Qual é a taxa geral de falhas das máquinas no período analisado?**

In [87]:
df.head(3)

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0


In [88]:
# Aqui vamos criar uma vriavel para armazenar o percentual de falhas na máquina, 
# e renomear os valores 0 e 1 para "Não Falhou" e "Falhou", respectivamente.
percentual_falhas = df["Machine failure"].value_counts() / 100
percentual_falhas = percentual_falhas.rename({0: "Não Falhou", 1: "Falhou"})

# Aaqui vamos criar dois gráficos de barras, um na orientação vertical e outro na horizontal, para visualizar a distribuição de falhas na máquina.
fig = px.bar(percentual_falhas,
             x=percentual_falhas.index, 
             y=percentual_falhas.values, 
             labels={"x": "Falha na Máquina", "y": "Contagem"}, 
             title="Distribuição de Falhas na Máquina",
             text=percentual_falhas.values)
fig.update_traces(
                    marker_color=["skyblue", "mediumaquamarine"],
                    texttemplate="%{text:.2f}%"
                    )  
fig.show()

# Agora vamos criar o gráfico de barras na orientação horizontal.
fig = px.bar(
            percentual_falhas,
            orientation='h', 
            x=percentual_falhas.values,
            y=percentual_falhas.index,
            labels={"x": "Contagem", "y": "Falha na Máquina"},
            title="Distribuição de Falhas na Máquina (Orientação Horizontal)",
            text=percentual_falhas.values
            )
fig.update_traces(
                    marker_color=["skyblue", "mediumaquamarine"],
                    )  
fig.show()

**Insight:** A taxa geral de falhas foi de **3,39%**, com 339 falhas em 10.000 registros. Os outros 9.661 registros não apresentaram falha. Como a base é sintética e não informa duração real de operação ou número de máquinas, esse percentual deve ser interpretado como a taxa observada no conjunto analisado, e não como uma estimativa universal da fábrica.

**Distribuição:**

- **Registros analisados:** 10.000.
- **Registros com falha:** 339 (3,39%).
- **Registros sem falha:** 9.661 (96,61%).

### Pergunta 2
**Quais tipos de falha (TWF, HDF, PWF, OSF, RNF) são mais frequentes?**

In [89]:
df.head(5)

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [90]:
# Agora vamos analisar as falhas mais frequentes na máquina. Para isso, 
# vamos somar os valores das colunas "TWF", "HDF", "PWF", "OSF" e "RNF", 
# que representam diferentes tipos de falhas, e ordenar os resultados em ordem decrescente, e transformar em percentual.

falhas_mais_frequentes = (
                            df[["TWF", "HDF", "PWF", "OSF", "RNF"]]
                            .sum()
                            .sort_values(ascending=False)
                            .reset_index()
                            .rename(columns={"index": "Tipo de Falha", 0: "Contagem"})
                            )
falhas_mais_frequentes["Percentual"] = ((falhas_mais_frequentes["Contagem"] / falhas_mais_frequentes["Contagem"].sum()) * 100).round(2)
fig = px.bar(
                falhas_mais_frequentes,
                x="Tipo de Falha",
                y="Contagem",
                labels={"Contagem": "Contagem de Falhas"},
                title="Falhas Mais Frequentes na Máquina",
                text="Percentual"
                )
fig.update_traces(
                    marker_color="mediumaquamarine",
                    texttemplate="%{text:.2f}%"
                    )
fig.show()  
falhas_mais_frequentes


,Tipo de Falha,Contagem,Percentual
0,HDF,115,30.83
1,OSF,98,26.27
2,PWF,95,25.47
3,TWF,46,12.33
4,RNF,19,5.09


**Insight:** *Aqui já podemos retirar um insight valioso, vimos que a máquina falhou 339 vezees.*

- *30.83% Das vezes que a máquina falhou foi por conta do HDF*
- *26.27% Das vezes que a máquina falhou foi por conta do OSF*
- *25.47% Das vezes que a máquina falhou foi por conta do PWF*

- **Somando os 3 da 82.57% das falhas, então precisamos aanalisar isso mais a fundo!**

- *Outras falham dão cerca de:* 17.42% das falhas, que é relativamente baixo, mas precisamos analisar.

### Pergunta 3
**Existe diferença na taxa de falha entre os diferentes tipos/qualidade de produto (L, M, H)?**

In [121]:
quantidade_pecas = df["Type"].value_counts()

resultado = (
    df.groupby("Type")
    .agg(
        Quantidade_de_Peças=("Type", "size"),
        Quantidade_de_Falhas=("Machine failure", "sum")
    )
    .reset_index()
    .rename(columns={"Type": "Tipo de Peça"})
)
resultado["Taxa de Falha (%)"] = (
    resultado["Quantidade_de_Falhas"] / resultado["Quantidade_de_Peças"] * 100
).round(2)
resultado = resultado.sort_values(by="Taxa de Falha (%)", ascending=False)

fig = px.bar(
    resultado,
    x="Tipo de Peça",
    y="Taxa de Falha (%)",
    labels={"Taxa de Falha (%)": "Taxa de Falha (%)"},
    title="Taxa de Falha por Tipo de Peça",
    text="Taxa de Falha (%)"
)
fig.update_traces(
    marker_color="mediumaquamarine",
    texttemplate="%{text:.2f}%"
)
fig.show()
resultado

,Tipo de Peça,Quantidade_de_Peças,Quantidade_de_Falhas,Taxa de Falha (%)
1,L,6000,235,3.92
2,M,2997,83,2.77
0,H,1003,21,2.09


In [92]:
falhas_por_tipo_de_peca = df.groupby("Type").agg(
    TWF=("TWF", "sum"),
    HDF=("HDF", "sum"),
    PWF=("PWF", "sum"),
    OSF=("OSF", "sum"))
falhas_por_tipo_de_peca["Total de Falhas"] = falhas_por_tipo_de_peca.sum(axis=1)
falhas_por_tipo_de_peca

,TWF,HDF,PWF,OSF,Total de Falhas
Type,,,,,
H,7,8,5,2,22
L,25,76,59,87,247
M,14,31,31,9,85


**Insight:** Quando comparamos as taxas, o tipo **L** apresentou o maior percentual de falhas (**3,92%**), seguido por **M (2,77%)** e **H (2,09%)**. A quantidade absoluta de falhas é maior em L também porque esse grupo é maior na base. O resultado indica uma diferença observada entre os tipos, mas não prova que a qualidade do produto seja a causa da falha.

### Pergunta 4
**Como a temperatura do ar e a temperatura do processo se relacionam com a ocorrência de falhas?**

In [108]:
fig = px.box(
    df,
    x="Machine failure",
    y="Air temperature [K]",
    color="Machine failure",
    title="Temperatura do Ar por Ocorrência de Falha"
)

fig.show()

df.groupby("Machine failure")[
    ["Air temperature [K]", "Process temperature [K]"]
].agg(["mean", "median", "min", "max"]).reset_index().rename(columns={"Machine failure": "Falha na Máquina"}).sort_values(by="Falha na Máquina", ascending=False)


Falha na Máquina Air temperature [K]                       \
                                  mean median    min    max   
1                1          300.886431  301.6  295.6  304.4   
0                0          299.973999  300.0  295.3  304.5   

  Process temperature [K]                       
                     mean median    min    max  
1              310.290265  310.4  306.1  313.7  
0              309.995570  310.0  305.7  313.8

**Insight:** *Vimos que o número não estão divididos em grupos, estão muito próximos, e o boxplot nos mostra que ambos são semelhantes.*

**Pontos:**
- *Não podemos dizer definitivamente que a temperatura do local e a tempertura da máquina não influenciam na falha da máquina.*
- *Mas se tivern uma diferença é relativamente muito baixa, precisamos analisar outras váriaveis.*

### Pergunta 5
**A diferença entre temperatura do processo e temperatura do ar (delta térmico) influencia falhas por dissipação de calor (HDF)?**

In [112]:
# O delta térmico mostra a diferença entre a temperatura do processo e a temperatura do ar.
df["Delta termico [K]"] = df["Process temperature [K]"] - df["Air temperature [K]"]

taxa_hdf = (
    df.groupby("HDF")
    .agg(
        Registros=("HDF", "size"),
        Falhas_HDF=("HDF", "sum"),
        Delta_medio=("Delta termico [K]", "mean"),
        Delta_mediano=("Delta termico [K]", "median")
    )
    .reset_index()
)
taxa_hdf["Taxa de HDF (%)"] = (taxa_hdf["Falhas_HDF"] / taxa_hdf["Registros"] * 100).round(2)
taxa_hdf["HDF"] = taxa_hdf["HDF"].map({0: "Não HDF", 1: "HDF"})

taxa_hdf

,HDF,Registros,Falhas_HDF,Delta_medio,Delta_mediano,Taxa de HDF (%)
0,Não HDF,9885,0,10.021254,9.8,0.0
1,HDF,115,115,8.227826,8.3,100.0


**Insight:** Os registros com HDF apresentaram delta térmico médio de **8,23 K**, abaixo dos **10,02 K** observados nos demais registros. Isso é compatível com menor dissipação de calor: quando a diferença entre processo e ambiente diminui, a temperatura do processo pode permanecer elevada. O delta é útil como alerta, mas deve ser acompanhado junto das temperaturas absolutas.

### Pergunta 6
**Qual a relação entre velocidade rotacional e torque em máquinas que falharam vs. que não falharam?**

In [123]:
fig = px.scatter(
    df,
    x="Rotational speed [rpm]",
    y="Torque [Nm]",
    color="Machine failure",
    labels={"Machine failure": "Falha na Máquina"},
    opacity=0.55,
    title="Velocidade Rotacional e Torque por Ocorrência de Falha"
)
fig.show()

comparacao_velocidade_torque = (
    df.groupby("Machine failure")[["Rotational speed [rpm]", "Torque [Nm]"]]
    .agg(["mean", "median"])
    .round(2)
    .T
)
comparacao_velocidade_torque.columns = ["Não falhou", "Falhou"]
comparacao_velocidade_torque

Não falhou   Falhou
Rotational speed [rpm] mean       1540.26  1496.49
                       median     1507.00  1365.00
Torque [Nm]            mean         39.63    50.17
                       median       39.90    53.70

**Insight:** As máquinas que falharam apresentaram torque médio maior (**50,17 Nm**) e velocidade rotacional média menor (**1.496 rpm**) do que as que não falharam (**39,63 Nm** e **1.540 rpm**). O padrão sugere operação com maior esforço mecânico e menor rotação, especialmente relevante para investigar sobrecarga e potência.

### Pergunta 7
**O desgaste da ferramenta (tool wear) está associado a maior probabilidade de falha?**

In [114]:
comparacao_desgaste = (
    df.groupby("Machine failure")["Tool wear [min]"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
    .rename(index={0: "Não falhou", 1: "Falhou"})
)

fig = px.box(
    df,
    x="Machine failure",
    y="Tool wear [min]",
    color="Machine failure",
    labels={"Machine failure": "Falha na Máquina", "Tool wear [min]": "Desgaste da Ferramenta (min)"},
    title="Desgaste da Ferramenta por Ocorrência de Falha"
)
fig.show()
comparacao_desgaste

,count,mean,median,min,max
Machine failure,,,,,
Não falhou,9661,106.69,107.0,0,246
Falhou,339,143.78,165.0,0,253


**Insight:** O desgaste médio foi de **143,78 min** nas peças que falharam, contra **106,69 min** nas que não falharam. A diferença indica associação entre desgaste e falha, embora o desgaste isolado não seja suficiente para explicar todos os casos.

### Pergunta 8
**Existe um limiar de desgaste da ferramenta a partir do qual o risco de falha aumenta significativamente?**

In [115]:
limites_desgaste = [0, 50, 100, 150, 200, np.inf]
rotulos_desgaste = ["0-49", "50-99", "100-149", "150-199", "200+"]
df["Faixa de desgaste"] = pd.cut(
    df["Tool wear [min]"],
    bins=limites_desgaste,
    labels=rotulos_desgaste,
    right=False
)

risco_por_desgaste = (
    df.groupby("Faixa de desgaste", observed=False)
    .agg(Registros=("Machine failure", "size"), Falhas=("Machine failure", "sum"))
    .reset_index()
)
risco_por_desgaste["Taxa de falha (%)"] = (
    risco_por_desgaste["Falhas"] / risco_por_desgaste["Registros"] * 100
).round(2)

fig = px.bar(
    risco_por_desgaste,
    x="Faixa de desgaste",
    y="Taxa de falha (%)",
    text="Taxa de falha (%)",
    labels={"Taxa de falha (%)": "Taxa de falha (%)"},
    title="Taxa de Falha por Faixa de Desgaste da Ferramenta"
)
fig.show()
risco_por_desgaste

,Faixa de desgaste,Registros,Falhas,Taxa de falha (%)
0,0-49,2349,52,2.21
1,50-99,2271,51,2.25
2,100-149,2290,52,2.27
3,150-199,2289,61,2.66
4,200+,801,123,15.36


**Insight:** A partir de **200 min** de desgaste, a taxa de falha sobe para **15,36%**, enquanto permanece entre 2,21% e 2,66% nas faixas anteriores. Esse ponto pode ser usado como limiar operacional inicial para inspeção ou troca preventiva, devendo ser recalibrado com dados reais de custo e vida útil.

### Pergunta 9
**Quais combinações de variáveis operacionais mais aparecem em falhas por sobrecarga (OSF)?**

In [116]:
osf = df[df["OSF"] == 1].copy()
osf["Faixa de torque"] = pd.qcut(osf["Torque [Nm]"], q=3, duplicates="drop")
osf["Faixa de velocidade"] = pd.qcut(osf["Rotational speed [rpm]"], q=3, duplicates="drop")
osf["Faixa de desgaste"] = pd.qcut(osf["Tool wear [min]"], q=3, duplicates="drop")

combinacoes_osf = (
    osf.groupby(["Faixa de torque", "Faixa de velocidade", "Faixa de desgaste"], observed=False)
    .size()
    .reset_index(name="Quantidade")
    .sort_values("Quantidade", ascending=False)
    .head(10)
)
combinacoes_osf

,Faixa de torque,Faixa de velocidade,Faixa de desgaste,Quantidade
8,"(46.299, 55.467]","(1372.667, 1515.0]","(213.0, 253.0]",15
18,"(60.833, 75.4]","(1180.999, 1333.667]","(171.999, 202.0]",14
13,"(55.467, 60.833]","(1333.667, 1372.667]","(202.0, 213.0]",9
12,"(55.467, 60.833]","(1333.667, 1372.667]","(171.999, 202.0]",7
7,"(46.299, 55.467]","(1372.667, 1515.0]","(202.0, 213.0]",6
19,"(60.833, 75.4]","(1180.999, 1333.667]","(202.0, 213.0]",6
15,"(55.467, 60.833]","(1372.667, 1515.0]","(171.999, 202.0]",5
5,"(46.299, 55.467]","(1333.667, 1372.667]","(213.0, 253.0]",5
21,"(60.833, 75.4]","(1333.667, 1372.667]","(171.999, 202.0]",4
20,"(60.833, 75.4]","(1180.999, 1333.667]","(213.0, 253.0]",4


**Insight:** Nos registros com OSF, as combinações mais frequentes concentram torque alto, velocidade baixa ou intermediária e desgaste elevado. O grupo mais frequente combinou torque entre 46,3 e 55,5 Nm, rotação entre 1.373 e 1.515 rpm e desgaste acima de 213 min. Essas faixas são úteis para criar alertas combinados, mas não representam um limite causal.

### Pergunta 10
**As falhas aleatórias (RNF) apresentam algum padrão identificável ou são realmente imprevisíveis?**

In [117]:
comparacao_rnf = (
    df.groupby("RNF")
    .agg(
        Registros=("RNF", "size"),
        Falhas_gerais=("Machine failure", "sum"),
        Temperatura_ar=("Air temperature [K]", "mean"),
        Temperatura_processo=("Process temperature [K]", "mean"),
        Velocidade=("Rotational speed [rpm]", "mean"),
        Torque=("Torque [Nm]", "mean"),
        Desgaste=("Tool wear [min]", "mean")
    )
    .reset_index()
)
comparacao_rnf["Taxa de falha geral (%)"] = (
    comparacao_rnf["Falhas_gerais"] / comparacao_rnf["Registros"] * 100
).round(2)
comparacao_rnf["RNF"] = comparacao_rnf["RNF"].map({0: "Não RNF", 1: "RNF"})
comparacao_rnf.round(2)

,RNF,Registros,Falhas_gerais,Temperatura_ar,Temperatura_processo,Velocidade,Torque,Desgaste,Taxa de falha geral (%)
0,Não RNF,9981,338,300.00,310.00,1538.88,39.98,107.92,3.39
1,RNF,19,1,300.82,310.76,1485.00,43.67,124.47,5.26


**Insight:** RNF aparece em apenas **19 registros**, dos quais 1 teve falha geral. A taxa foi de 5,26% nos registros com RNF, contra 3,39% nos demais, mas a quantidade é pequena demais para confirmar um padrão. Por isso, esse tipo deve ser tratado como evento de baixa previsibilidade e monitorado continuamente.

### Pergunta 11
**Se cada falha gera, em média, X horas de máquina parada, qual o impacto estimado no período analisado?**

In [118]:
# O valor pode ser ajustado conforme o histórico real de duração das paradas.
horas_parada_por_falha = 1
numero_de_falhas = int(df["Machine failure"].sum())
horas_paradas_estimadas = numero_de_falhas * horas_parada_por_falha

impacto_paradas = pd.DataFrame({
    "Indicador": ["Falhas registradas", "Horas médias por falha", "Horas de parada estimadas"],
    "Valor": [numero_de_falhas, horas_parada_por_falha, horas_paradas_estimadas]
})
impacto_paradas

,Indicador,Valor
0,Falhas registradas,339
1,Horas médias por falha,1
2,Horas de parada estimadas,339


**Insight:** Como o valor de X não foi informado, o notebook deixa `horas_parada_por_falha` como parâmetro. Com o valor padrão de 1 hora, as 339 falhas representam **339 horas estimadas de parada**. Para obter o impacto real, basta substituir esse parâmetro pela média histórica e, depois, multiplicar pelas perdas de produção e pelos custos de manutenção.

### Pergunta 12
**Quais variáveis têm maior correlação com a ocorrência de falha geral?**

In [119]:
colunas_numericas = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "Machine failure"
]
correlacoes = (
    df[colunas_numericas]
    .corr()["Machine failure"]
    .drop("Machine failure")
    .sort_values(key=abs, ascending=False)
    .rename("Correlação com falha")
    .round(3)
    .to_frame()
)

fig = px.bar(
    correlacoes.reset_index(),
    x="Correlação com falha",
    y="index",
    orientation="h",
    text="Correlação com falha",
    labels={"index": "Variável"},
    title="Correlação das Variáveis Operacionais com a Falha Geral"
)
fig.show()
correlacoes

,Correlação com falha
Torque [Nm],0.191
Tool wear [min],0.105
Air temperature [K],0.083
Rotational speed [rpm],-0.044
Process temperature [K],0.036


**Insight:** A correlação linear foi maior para **Torque (0,191)**, seguido por **Tool wear (0,105)** e **temperatura do ar (0,083)**. São associações fracas a moderadas e não implicam causalidade; correlação também não captura bem regras combinadas entre variáveis. Ainda assim, o resultado reforça o uso de torque e desgaste como variáveis prioritárias em alertas.

### Pergunta 13
**É possível segmentar as máquinas em grupos de risco (baixo, médio, alto) com base nas variáveis disponíveis?**

In [120]:
variaveis_risco = ["Torque [Nm]", "Tool wear [min]", "Air temperature [K]", "Rotational speed [rpm]"]

# Cada variável recebe um ponto quando está acima do percentil 75 da própria base.
for variavel in variaveis_risco:
    limite = df[variavel].quantile(0.75)
    df[f"Alerta {variavel}"] = (df[variavel] >= limite).astype(int)

df["Pontuacao de risco"] = df[[f"Alerta {variavel}" for variavel in variaveis_risco]].sum(axis=1)
df["Grupo de risco"] = pd.cut(
    df["Pontuacao de risco"],
    bins=[-1, 1, 2, 4],
    labels=["Baixo", "Médio", "Alto"]
)

segmentacao_risco = (
    df.groupby("Grupo de risco", observed=False)
    .agg(Registros=("Machine failure", "size"), Falhas=("Machine failure", "sum"))
    .reset_index()
)
segmentacao_risco["Taxa de falha (%)"] = (
    segmentacao_risco["Falhas"] / segmentacao_risco["Registros"] * 100
).round(2)

fig = px.bar(
    segmentacao_risco,
    x="Grupo de risco",
    y="Taxa de falha (%)",
    text="Taxa de falha (%)",
    category_orders={"Grupo de risco": ["Baixo", "Médio", "Alto"]},
    labels={"Taxa de falha (%)": "Taxa de falha (%)"},
    title="Taxa de Falha por Grupo de Risco Operacional"
)
fig.show()
segmentacao_risco

,Grupo de risco,Registros,Falhas,Taxa de falha (%)
0,Baixo,7404,85,1.15
1,Médio,2251,205,9.11
2,Alto,345,49,14.20


**Insight:** A pontuação operacional separou os registros em grupos com taxas de falha de **1,15% (baixo)**, **9,11% (médio)** e **14,20% (alto)**. A segmentação é um filtro inicial, baseado em valores acima do percentil 75, e não um modelo preditivo validado. Ela pode apoiar a priorização de inspeções enquanto a empresa reúne mais dados.

## 9. Perguntas Finais e Recomendações

### Pergunta 14 — Quais recomendações de manutenção preventiva podem ser feitas com base nos padrões encontrados?

1. Criar um alerta de inspeção para ferramentas com desgaste a partir de **200 min**, faixa em que a taxa observada de falha chegou a 15,36%.
2. Monitorar torque e velocidade em conjunto, priorizando situações de torque elevado com rotação baixa ou intermediária.
3. Acompanhar o delta térmico e as temperaturas absolutas para identificar condições compatíveis com falha de dissipação de calor.
4. Usar a pontuação de risco para priorizar inspeções: registros no grupo alto devem ser avaliados antes dos grupos médio e baixo.
5. Registrar duração, causa e custo de cada parada. Essas informações permitirão substituir o cenário parametrizado de impacto por uma estimativa financeira real.

### Pergunta 15 — Se a empresa pudesse monitorar de perto apenas 2 ou 3 variáveis para antecipar falhas, quais seriam e por quê?

As prioridades seriam **Torque**, **Tool wear** e o **delta térmico**. Torque apresentou a maior correlação com a falha geral e também foi mais alto entre as máquinas que falharam. O desgaste mostrou aumento forte de risco na faixa de 200 min ou mais. Já o delta térmico ajuda a identificar condições associadas à falha de dissipação de calor. A rotação deve ser usada como variável complementar, principalmente na interpretação do torque.


## 10. Conclusão Geral

A análise dos 10.000 registros encontrou uma taxa geral de falhas de **3,39%**. HDF, OSF e PWF concentraram a maior parte das ocorrências, enquanto torque e desgaste foram as variáveis mais associadas à falha geral. O desgaste acima de 200 min apresentou aumento expressivo da taxa de falha, e o delta térmico menor apareceu associado aos casos de HDF. A segmentação operacional também criou grupos com diferentes níveis de risco, úteis para priorizar inspeções. A recomendação principal é implantar alertas combinados de torque, desgaste e condição térmica, acompanhados de registros reais de paradas para validar e aprimorar os limites.

